In [3]:
import os
import cv2 as cv
import numpy as np
from mtcnn.mtcnn import MTCNN
from PIL import Image
from keras_facenet import FaceNet

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import pickle
import warnings
warnings.filterwarnings("ignore")

In [4]:
# ---- Config ----
TARGET_SIZE = (160, 160)        # FaceNet Default Input
ALLOWED_EXT = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
SAVED_MODEL_PATH = 'svm_face_model.pkl'

In [5]:
class FACELOADING:
    def __init__(self, directory, target_size=TARGET_SIZE):
        self.directory = directory
        self.target_size = target_size
        self.detector = MTCNN()
        self.X = []
        self.Y = []

    # ------------------------ 
    # IMAGE READ + JPG CONVERT
    # ------------------------
    def read_and_convert(self, filepath):
        try:
            img = Image.open(filepath).convert("RGB")
        except Exception as e:
            print(f"❌ Bad image removed: {filepath}")
            try: os.remove(filepath)
            except: pass
            return None, None

        ext = os.path.splitext(filepath)[1].lower()

        # non-jpg → convert to .jpg
        if ext not in ('.jpg', '.jpeg'):
            new_path = os.path.splitext(filepath)[0] + ".jpg"
            try:
                img.save(new_path, "JPEG", quality=95)
                if new_path != filepath:
                    os.remove(filepath)
                filepath = new_path
            except:
                return None, None

        img_np = np.array(img)
        img_bgr = cv.cvtColor(img_np, cv.COLOR_RGB2BGR)
        return img_bgr, filepath

    # ------------------------ 
    # FACE EXTRACT
    # ------------------------
    def extract_face(self, filepath):
        img_bgr, final_path = self.read_and_convert(filepath)
        if img_bgr is None:
            return None

        img_rgb = cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB)
        faces = self.detector.detect_faces(img_rgb)

        if not faces:
            print(f"⚠️ No face found: {final_path}")
            return None

        x, y, w, h = faces[0]['box']
        x, y = abs(x), abs(y)
        x2, y2 = x + w, y + h

        face = img_rgb[y:y2, x:x2]

        if face.size == 0:
            return None

        face_resized = cv.resize(face, self.target_size)
        return face_resized

    # ------------------------ 
    # LOAD CLASS FOLDER IMAGES
    # ------------------------
    def load_faces_from_class(self, class_dir):
        faces = []
        for fname in os.listdir(class_dir):
            fullpath = os.path.join(class_dir, fname)
            if not os.path.isfile(fullpath):
                continue

            face = self.extract_face(fullpath)
            if face is not None:
                faces.append(face)
        return faces

    # ------------------------ 
    # LOAD ALL CLASSES
    # ------------------------
    def load_classes(self):
        for sub in sorted(os.listdir(self.directory)):
            subpath = os.path.join(self.directory, sub)
            if not os.path.isdir(subpath):
                continue

            faces = self.load_faces_from_class(subpath)
            labels = [sub] * len(faces)

            print(f"✅ Loaded {len(faces)} faces for '{sub}'")
            self.X.extend(faces)
            self.Y.extend(labels)

        return np.asarray(self.X), np.asarray(self.Y)

    # ------------------------ 
    # SHOW SAMPLES (matplotlib)
    # ------------------------
    def plot_samples(self, limit=9):
        import math
        if len(self.X) == 0:
            print("No images to show.")
            return

        n = min(limit, len(self.X))
        cols = 3
        rows = math.ceil(n / cols)

        plt.figure(figsize=(cols * 3, rows * 3))
        for i in range(n):
            plt.subplot(rows, cols, i + 1)
            plt.imshow(self.X[i])
            plt.axis('off')
        plt.show()


In [6]:
embedder = FaceNet()

def get_embedding(face_img):
    img = face_img.astype('float32')
    img = np.expand_dims(img, axis=0)
    emb = embedder.embeddings(img)
    return emb[0]

2025-11-20 18:41:22.833058: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [7]:
def train_svm_with_metrics(X_emb, Y_labels):
    encoder = LabelEncoder()
    Y_enc = encoder.fit_transform(Y_labels)

    X_train, X_test, Y_train, Y_test = train_test_split(
        X_emb, Y_enc, test_size=0.25, random_state=42, stratify=Y_enc
    )

    model = SVC(kernel='linear', probability=True)
    model.fit(X_train, Y_train)

    Y_pred = model.predict(X_test)

    print("\n📌 Classification Report:")
    print(classification_report(Y_test, Y_pred, target_names=encoder.classes_))

    # Confusion matrix plot
    cm = confusion_matrix(Y_test, Y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=encoder.classes_,
                yticklabels=encoder.classes_,
                cmap='Blues')
    plt.title("Confusion Matrix")
    plt.show()

    return model, encoder


In [8]:
def predict_image_from_path(model, encoder, image_path):
    detector = MTCNN()

    img = Image.open(image_path).convert("RGB")
    img_np = np.array(img)

    faces = detector.detect_faces(img_np)
    if not faces:
        print("No face found.")
        return None

    x, y, w, h = faces[0]['box']
    x, y = abs(x), abs(y)
    face = img_np[y:y+h, x:x+w]

    face_resized = cv.resize(face, TARGET_SIZE)
    emb = get_embedding(face_resized).reshape(1, -1)

    pred = model.predict(emb)
    name = encoder.inverse_transform(pred)[0]
    prob = np.max(model.predict_proba(emb))

    print(f"Prediction: {name}, Prob: {prob:.2f}")
    return name, prob


In [9]:
def save_model_and_encoder(model, encoder):
    with open(SAVED_MODEL_PATH, 'wb') as f:
        pickle.dump({'model': model, 'encoder': encoder}, f)
    print("💾 Model saved!")

def load_model_and_encoder():
    with open(SAVED_MODEL_PATH, 'rb') as f:
        data = pickle.load(f)
    return data['model'], data['encoder']


In [10]:
def run_pipeline(train_dir, test_image_path=None):
    loader = FACELOADING(train_dir)
    X_images, Y_labels = loader.load_classes()

    loader.plot_samples()

    print("⚙️ Generating embeddings...")
    EMBEDDED = np.array([get_embedding(img) for img in X_images])

    model, encoder = train_svm_with_metrics(EMBEDDED, Y_labels)

    save_model_and_encoder(model, encoder)

    if test_image_path:
        predict_image_from_path(model, encoder, test_image_path)

    return model, encoder

In [11]:
if __name__ == "__main__":
    train_dir = "dataset/train"
    test_image = "dataset/test/ronaldo/ronaldo.jpg"
    
    model, encoder = run_pipeline(train_dir, test_image_path=test_image, plot_samples=True)

TypeError: run_pipeline() got an unexpected keyword argument 'plot_samples'